## **E. LLM Based Chunking**

### What is it?
- LLM-based chunking uses a language model to intelligently determine
where to split text based on semantic understanding. The LLM
analyzes the document and decides optimal chunk boundaries,
potentially generating summaries or extracting key information.
- This is like having an intelligent editor who understands the content
and knows exactly where topics begin and end.

> Advantages:
- Most intelligent and context-aware splitting
- Can add contextual summaries to chunks
- Understands semantic boundaries better than rules
- Can adapt to different document types
- Improves retrieval quality significantly
- Can extract key information

> Disadvantages:
- Very expensive (LLM API calls for every chunk)
- Slowest processing time
- Requires API access and costs money
- Not deterministic (results may vary)
- Overkill for simple documents
- Latency issues for large documents
- Complex to implement and maintain

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate

In [4]:
load_dotenv()

True

In [5]:
text= """Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structule.
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future. """

In [6]:
## Pydantic Class for Structure Output
class Chunk(BaseModel):
    chunk_text: str
    summary: str

class Chunker(BaseModel):
    chunks: list[Chunk]

In [7]:
model= ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

llm_chunker= model.with_structured_output(schema= Chunker)

In [8]:
prompt = ChatPromptTemplate(messages=[
    ( "system",
    """You are an expert Text Chunker that splits the given text and outputs them as a
    list of strings. You understand the natural topic boundaries of text and
    also do not change the existing text. You just split the text where ever applicable.
    Once you create the chunk, you also generate a 1-2 line summary of the chunk also"""),
    ( "human",
    "Split the given text into chunks\nText: {text}")
], input_variables=["text"])

In [9]:
# chunking through 11m

model_chain = prompt | llm_chunker

response = model_chain.invoke({"text": text})

In [10]:
response

Chunker(chunks=[Chunk(chunk_text="Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structule.", summary='This section discusses advancements in artificial intelligence, machine learning, deep learning, and neural network structures.'), Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.', summary='This part covers key principles of Italian pasta cooking, including ingredient selection and preparation techniques.'), Chunk(chunk_text='Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to m

In [22]:
len(response.chunks)

3

In [12]:
response.chunks[0].chunk_text

"Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structule."

In [29]:
for i in range(0, len(response.chunks)):
    print(f"Chunk {i+1}: {response.chunks[i].chunk_text}")
    print(f"Summary {i+1}: {response.chunks[i].summary}\n")

Chunk 1: Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structule.
Summary 1: This section discusses advancements in artificial intelligence, machine learning, deep learning, and neural network structures.

Chunk 2: The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.
Summary 2: This part covers key principles of Italian pasta cooking, including ingredient selection and preparation techniques.

Chunk 3: Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to melt at unprecedented rates. Scientists warn tha